# Governance, Evaluation & Traceability

**Track:** Applied Agent Engineering Foundations  
**Module:** Governance, Evaluation & Traceability  
**Environment:** SageMaker Jupyter Notebook

## What learners will learn
1. Add policy guardrails before model/tool execution.
2. Create a lightweight evaluation set.
3. Capture outputs, latency, and audit-friendly logs.
4. Separate safe questions from blocked questions.
5. Build a mini evaluation harness for enterprise agent quality.

> **Explain to learners:** Governance is not an afterthought. It is part of the runtime architecture.


## Recommended use of this notebook

learners already understand:
- basic prompting
- simple tool routing
- RAG basics

This notebook focuses on **reliability, safety, and observability**, not on building the model itself.


In [2]:

# Uncomment if needed
# %pip install -q boto3 pandas

import json
import time
from datetime import datetime

import boto3
import pandas as pd

BEDROCK_TEXT_MODEL = "amazon.nova-lite-v1:0"
AWS_REGION = boto3.Session().region_name or "us-east-1"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)


Note: you may need to restart the kernel to use updated packages.


## Step 1 — Define a simple enterprise assistant

We start with a lightweight assistant so that we can focus on governance patterns.


In [3]:

POLICY_STORE = {
    "leave policy": "Employees can apply planned leave through the HR portal with manager approval.",
    "remote work": "Remote work is allowed for approved roles and depends on team policy.",
    "contractor policy": "Contractors must follow the contract-specific access and compliance rules."
}

def policy_lookup(query: str) -> str:
    q = query.lower()
    for key, value in POLICY_STORE.items():
        if key in q:
            return value
    return "I don't know."

def answer_with_policy(query: str) -> str:
    # In M3, a deliberately simple answer path makes governance easier to observe.
    return policy_lookup(query)


## Step 2 — Add a pre-execution guardrail

This guardrail blocks salary and compensation questions.
That mirrors the intent already present in your original code and keeps the policy visible to learners.


In [4]:

BLOCKED_KEYWORDS = [
    "salary",
    "compensation",
    "payroll",
    "pay band",
    "salary hike",
    "ctc",
    "employee salary"
]

def guardrail_check(query: str) -> None:
    lowered = query.lower()
    if any(keyword in lowered for keyword in BLOCKED_KEYWORDS):
        raise ValueError(
            "Policy block: I cannot help with salary, compensation, payroll, or confidential pay information."
        )


## Step 3 — Wrap the assistant with governance logging

We will record:
- timestamp
- input
- output
- blocked flag
- latency
- status


In [5]:

audit_log = []

def governed_run(query: str) -> dict:
    start = time.time()
    timestamp = datetime.utcnow().isoformat()

    try:
        guardrail_check(query)
        answer = answer_with_policy(query)
        status = "ok"
        blocked = False
        error = None
    except Exception as e:
        answer = None
        status = "blocked_or_error"
        blocked = True
        error = str(e)

    latency_ms = round((time.time() - start) * 1000, 2)

    record = {
        "timestamp_utc": timestamp,
        "query": query,
        "answer": answer,
        "status": status,
        "blocked": blocked,
        "error": error,
        "latency_ms": latency_ms
    }

    audit_log.append(record)
    return record


In [6]:

tests = [
    "What is the leave policy?",
    "What is the remote work policy?",
    "Show me employee salary details.",
    "What is the contractor policy?"
]

for q in tests:
    print(governed_run(q))


{'timestamp_utc': '2026-04-19T14:19:58.149330', 'query': 'What is the leave policy?', 'answer': 'Employees can apply planned leave through the HR portal with manager approval.', 'status': 'ok', 'blocked': False, 'error': None, 'latency_ms': 0.13}
{'timestamp_utc': '2026-04-19T14:19:58.149411', 'query': 'What is the remote work policy?', 'answer': 'Remote work is allowed for approved roles and depends on team policy.', 'status': 'ok', 'blocked': False, 'error': None, 'latency_ms': 0.01}
{'timestamp_utc': '2026-04-19T14:19:58.149442', 'query': 'Show me employee salary details.', 'answer': None, 'status': 'blocked_or_error', 'blocked': True, 'error': 'Policy block: I cannot help with salary, compensation, payroll, or confidential pay information.', 'latency_ms': 0.01}
{'timestamp_utc': '2026-04-19T14:19:58.149469', 'query': 'What is the contractor policy?', 'answer': 'Contractors must follow the contract-specific access and compliance rules.', 'status': 'ok', 'blocked': False, 'error': No

/tmp/ipykernel_16261/1813287798.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()


## Step 4 — Turn logs into a table

**Explain to learners:** Audit trails help with:
- debugging
- compliance review
- incident triage
- stakeholder trust


In [7]:

audit_df = pd.DataFrame(audit_log)
audit_df


,timestamp_utc,query,answer,status,blocked,error,latency_ms
0,2026-04-19T14:19:58.149330,What is the leave policy?,Employees can apply planned leave through the ...,ok,False,None,0.13
1,2026-04-19T14:19:58.149411,What is the remote work policy?,Remote work is allowed for approved roles and ...,ok,False,None,0.01
2,2026-04-19T14:19:58.149442,Show me employee salary details.,None,blocked_or_error,True,"Policy block: I cannot help with salary, compe...",0.01
3,2026-04-19T14:19:58.149469,What is the contractor policy?,Contractors must follow the contract-specific ...,ok,False,None,0.01


## Step 5 — Add a mini evaluation set

We will classify a few test cases by expected behavior:
- answer
- refuse/block
- unknown


In [8]:

evaluation_cases = [
    {
        "query": "What is the leave policy?",
        "expected_behavior": "answer"
    },
    {
        "query": "What is the remote work policy?",
        "expected_behavior": "answer"
    },
    {
        "query": "Show me employee salary details.",
        "expected_behavior": "block"
    },
    {
        "query": "What is the cafeteria menu policy?",
        "expected_behavior": "unknown"
    }
]

pd.DataFrame(evaluation_cases)


,query,expected_behavior
0,What is the leave policy?,answer
1,What is the remote work policy?,answer
2,Show me employee salary details.,block
3,What is the cafeteria menu policy?,unknown


In [9]:

def evaluate_case(case: dict) -> dict:
    result = governed_run(case["query"])

    if result["blocked"]:
        observed_behavior = "block"
    elif result["answer"] == "I don't know.":
        observed_behavior = "unknown"
    else:
        observed_behavior = "answer"

    return {
        "query": case["query"],
        "expected_behavior": case["expected_behavior"],
        "observed_behavior": observed_behavior,
        "pass": observed_behavior == case["expected_behavior"],
        "latency_ms": result["latency_ms"]
    }

evaluation_results = [evaluate_case(case) for case in evaluation_cases]
eval_df = pd.DataFrame(evaluation_results)
eval_df


/tmp/ipykernel_16261/1813287798.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().isoformat()


,query,expected_behavior,observed_behavior,pass,latency_ms
0,What is the leave policy?,answer,answer,True,0.12
1,What is the remote work policy?,answer,answer,True,0.01
2,Show me employee salary details.,block,block,True,0.01
3,What is the cafeteria menu policy?,unknown,unknown,True,0.01


## Step 6 — Summarize evaluation metrics

We keep the metrics intentionally simple for classroom clarity.


In [10]:

summary = {
    "total_cases": len(eval_df),
    "passes": int(eval_df["pass"].sum()),
    "pass_rate": round(float(eval_df["pass"].mean()) * 100, 2),
    "avg_latency_ms": round(float(eval_df["latency_ms"].mean()), 2)
}

summary


{'total_cases': 4, 'passes': 4, 'pass_rate': 100.0, 'avg_latency_ms': 0.04}

## Step 7 — Track prompt and system drift

Prompt drift can change model behavior even when the retrieved context stays the same.

For classroom discussion, compare two modes:
- **strict mode**: answer only from retrieved policy context
- **loose mode**: be helpful and fill gaps with a reasonable inference

This gives a much better governance example because the same retrieved context can produce:
- a safe refusal in strict mode
- a plausible but risky invented answer in loose mode


In [11]:

STRICT_SYSTEM_PROMPT = (
    "You are a careful enterprise assistant. "
    "Use only the retrieved policy context. "
    "Do not invent missing policy details."
)

LOOSE_SYSTEM_PROMPT = (
    "You are a helpful enterprise assistant. "
    "Use the retrieved policy context first, but if some details are missing, "
    "give a best-practice business answer and clearly label it as an inference."
)

print("Strict mode note:", STRICT_SYSTEM_PROMPT)
print("Loose mode note:", LOOSE_SYSTEM_PROMPT)
print("\nInstructor note: the loose prompt is intentionally riskier so learners can see why governance matters.")


Strict mode note: Answer only from known policy content. If not found, say I don't know.
Loose mode note: Be helpful and answer as best as you can.

Instructor exercise: ask learners why the stricter version is safer for enterprise workflows.


## Step 8 — Compare strict vs loose RAG behavior with Bedrock

Now we use the same retrieved policy context and send it to Bedrock in **two different prompt modes**.

### Why this is a better example
In the earlier version, both prompts still forced the model to say *I don't know*, so the outputs looked the same.

In this version:
- **strict mode** must stay inside the retrieved context
- **loose mode** is allowed to make a helpful inference when the context is incomplete

### Suggested classroom flow
Run these two queries:
1. a **known** query that is answerable from policy text
2. an **unknown** query that is **not** answerable from policy text

This lets learners see how prompt style changes enterprise risk.

Change `known_query` and `unknown_query` and run the cell again.


In [ ]:

def retrieve_policy_context(query: str, top_k: int = 2) -> list[dict]:
    query_lower = query.lower()
    scored = []

    for key, value in POLICY_STORE.items():
        score = 0
        for token in query_lower.split():
            token = token.strip("?.!,")
            if token and token in key.lower():
                score += 2
            if token and token in value.lower():
                score += 1
        if key in query_lower:
            score += 3

        scored.append({
            "policy_key": key,
            "policy_text": value,
            "score": score
        })

    scored = sorted(scored, key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


def build_rag_context(context_rows: list[dict]) -> str:
    blocks = []
    for i, row in enumerate(context_rows, start=1):
        blocks.append(f"[Source {i}] {row['policy_key']}\n{row['policy_text']}")
    return "\n\n".join(blocks)


def call_bedrock_rag(mode: str, system_prompt: str, query: str, context_text: str) -> str:
    if mode == "strict_prompt":
        user_instruction = (
            "Answer the question using only the retrieved policy context. "
            "If the answer is not in the context, reply exactly with: "
            "I don't know based on the retrieved policy context."
        )
        temperature = 0.0
    else:
        user_instruction = (
            "Answer the question using the retrieved policy context first. "
            "If the context is incomplete, give the most likely business answer and start the final sentence with "
            "'Inference:' so the learner can see that it is not grounded in policy."
        )
        temperature = 0.3

    response = bedrock_runtime.converse(
        modelId=BEDROCK_TEXT_MODEL,
        system=[{"text": system_prompt}],
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "text": (
                            f"{user_instruction}\n\n"
                            f"Retrieved context:\n{context_text}\n\n"
                            f"Question: {query}"
                        )
                    }
                ],
            }
        ],
        inferenceConfig={
            "maxTokens": 200,
            "temperature": temperature,
        },
    )

    content = response.get("output", {}).get("message", {}).get("content", [])
    text_parts = [item.get("text", "") for item in content if isinstance(item, dict) and "text" in item]
    return " ".join(text_parts).strip()


known_query = "How do employees apply planned leave?"
unknown_query = "How many casual leave days do employees get per year?"

comparison_rows = []

for query_label, comparison_query in [
    ("known_query", known_query),
    ("unknown_query", unknown_query),
]:
    retrieved_rows = retrieve_policy_context(comparison_query, top_k=2)
    retrieved_df = pd.DataFrame(retrieved_rows)
    context_text = build_rag_context(retrieved_rows)

    print(f"\nRetrieved policy rows for {query_label}:")
    with pd.option_context('display.max_columns', None,  # Show all columns
                       'display.max_colwidth', None, # Show full column content
                       'display.width', None):       # Auto-detect width
        display(retrieved_df)

    strict_answer = call_bedrock_rag(
        mode="strict_prompt",
        system_prompt=STRICT_SYSTEM_PROMPT,
        query=comparison_query,
        context_text=context_text,
    )

    loose_answer = call_bedrock_rag(
        mode="loose_prompt",
        system_prompt=LOOSE_SYSTEM_PROMPT,
        query=comparison_query,
        context_text=context_text,
    )

    comparison_rows.extend([
        {
            "query_label": query_label,
            "mode": "strict_prompt",
            "query": comparison_query,
            "answer": strict_answer,
        },
        {
            "query_label": query_label,
            "mode": "loose_prompt",
            "query": comparison_query,
            "answer": loose_answer,
        },
    ])

comparison_df = pd.DataFrame(comparison_rows)
print("\nStrict vs loose prompt comparison:")
with pd.option_context('display.max_columns', None,  # Show all columns
                       'display.max_colwidth', None, # Show full column content
                       'display.width', None):       # Auto-detect width
    display(comparison_df)

print(
    "\nInstructor note: for the unknown query, strict mode should refuse from missing context, "
    "while loose mode may produce an inferred answer. That difference is the governance lesson."
)


## Step 9 — Prepare audit artifacts as DataFrames

For classroom use in SageMaker, it is easier to inspect the results directly in notebook tables.

This keeps the workflow simple:
- no local file export
- easy to read during class
- easy for learners to modify and rerun

In [ ]:
audit_artifact_df = audit_df.copy()
evaluation_artifact_df = eval_df.copy()

print("Audit artifact table:")
display(audit_artifact_df)

print("Evaluation artifact table:")
display(evaluation_artifact_df)

## Mini-hack — Build one governance improvement

### Minimum requirements
1. Add one more blocked policy topic.
2. Add at least 3 new evaluation cases.
3. Show the updated pass rate.
4. Export the new audit log.

### Good mini-hack ideas
- block PII requests
- block internal financial data
- add a “needs human review” category
- add severity tags to the audit log


In [ ]:

# --- LEARNER WORK AREA ---
# Suggested extension:
# 1. Create a new list called HUMAN_REVIEW_TOPICS.
# 2. Tag certain queries as "needs_human_review" instead of answer/block.
# 3. Update evaluation logic accordingly.


## Wrap-up

Learners should now understand:
- where guardrails sit in the stack
- why evaluation needs explicit test cases
- how traceability supports governance
- why logs are essential for enterprise deployment
